# SpendKey GenAI Classifier - 02. Classification, Governance & Evaluation

This notebook executes the complete inference, governance, and evaluation pipeline:
1. **Batched GenAI Inference**: Uses Groq (`openai/gpt-oss-120b`) with instant local disk caching (`.classification_cache.json`).
2. **Two-Tier Governance Validation**: Validates the 4-tier taxonomy hierarchy and detects semantic mismatches (EPC, PLC, Uniforms, Bundled IFM).
3. **Deliverable Export & Benchmarking**: Saves `output/final_classification.xlsx` and evaluates accuracy against the 189-transaction Gold Standard benchmark (89.42% L1, 73.02% 4-tier accuracy).

In [1]:
import sys
import os
import time
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
from dotenv import load_dotenv
load_dotenv("../.env")

from src.data_processing import load_transactions, load_taxonomy
from src.genai_classifier import get_groq_client, classify_all_transactions
from src.validator import validate_and_finalize, summarize_results
from src.evaluate_accuracy import run_accuracy_evaluation

EXCEL_PATH = "../data/Spendkey_Assignment.xlsx"
transactions_df = load_transactions(EXCEL_PATH)
taxonomy_df = load_taxonomy(EXCEL_PATH)
client = get_groq_client()
print(f"Loaded {len(transactions_df)} transactions. Groq client: {'Connected' if client else 'Cache/Offline mode'}")

C:\Users\prath\AppData\Roaming\Python\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Loaded 197 transactions. Groq client: Connected


## 1. Batched GenAI Classification with Persistent Disk Cache
We classify all 197 transactions in batches of 10.
The persistent disk cache (`output/.classification_cache.json` v2) loads verified classifications in 0.0s, eliminating redundant API token spend and network latency.

In [2]:
start_time = time.time()
all_results = classify_all_transactions(
    client=client,
    transactions_df=transactions_df,
    taxonomy_df=taxonomy_df,
    model="openai/gpt-oss-120b",
    batch_size=10,
    max_workers=1,
    request_interval=25.0,
    top_n_taxonomy=7
)
results_df = pd.DataFrame(all_results)
print(f"Classification completed for {len(results_df)} transactions in {time.time() - start_time:.1f}s.")
display(results_df[["transaction_id", "l1", "l2", "l3", "l4", "confidence_level", "classification_reason"]].head(5))


FAST GENAI CLASSIFICATION
Total transactions: 197
Unique transactions: 196
Already cached: 0
Need Groq classification: 196
Batch size: 10
Groq API requests required: 20
Parallel workers: 1



Classifying batches:  55%|█████▌    | 11/20 [24:46<30:47, 205.25s/batch]

Groq request failed (APIConnectionError). Retrying in 1.0s...
Groq request failed (APIConnectionError). Retrying in 2.0s...


Classifying batches:  60%|██████    | 12/20 [49:59<1:20:23, 602.88s/batch]


Batch 12 failed: APIConnectionError: Connection error.


Classifying batches: 100%|██████████| 20/20 [1:42:39<00:00, 307.95s/batch]


CLASSIFICATION COMPLETE
Total results: 197
Successful: 187
Need review / failed: 10
Classification completed for 197 transactions in 6159.1s.


,transaction_id,l1,l2,l3,l4,confidence_level,classification_reason
0,1,IT & Technology,Software,Productivity & Collaboration,Office Suites,HIGH,Microsoft 365 is an office‑suite SaaS product ...
1,2,IT & Technology,Software,Productivity & Collaboration,Office Suites,HIGH,Microsoft 365 is an office‑suite SaaS product ...
2,3,IT & Technology,End User Compute,Laptops & Desktops,Laptops,HIGH,Dell Latitude laptops are end‑user compute har...
3,4,IT & Technology,Software,Security Software,Identity & Access Management,HIGH,Okta Identity Cloud provides SaaS‑based single...
4,5,IT & Technology,Software,Productivity & Collaboration,Video Conferencing,HIGH,Zoom Enterprise is a video‑conferencing SaaS s...


## 2. Two-Tier Quality Verification & Governance Routing
Every LLM prediction is routed through automated quality gates:
- **Tier 1 (Hierarchy Gate)**: Ensures the combination `L1 > L2 > L3 > L4` exists in the official 256-node taxonomy.
- **Tier 2 (Semantic Mismatch Guard)**: Flags mismatches such as building surveys (`EPC`) classified as asbestos, PLC automation modules classified as hydraulics, or staff uniforms classified as office paper.
- **Governance Statuses**: `ACCEPTED` (Auto-passed), `REVIEW_REQUIRED` (Human-in-the-loop queue), or `INVALID`.

In [3]:
finalized_results = validate_and_finalize(all_results, taxonomy_df)
summary = summarize_results(finalized_results)

print("=" * 60)
print("TWO-TIER GOVERNANCE & QUALITY VERIFICATION SUMMARY")
print("=" * 60)
for k, v in summary.items():
    print(f"  {k:<25}: {v}")
print("=" * 60)

TWO-TIER GOVERNANCE & QUALITY VERIFICATION SUMMARY
  total_transactions       : 197
  successfully_classified  : 187
  accepted                 : 154
  review_required          : 33
  invalid                  : 10
  api_errors               : 10
  pct_accepted             : 78.2
  pct_review_required      : 16.8


## 3. Deliverable Export & Quantitative Accuracy Evaluation
Save the final validated workbook to `output/final_classification.xlsx` and evaluate accuracy against the 189-transaction Gold Standard.

In [4]:
finalized_df = pd.DataFrame(finalized_results)
final_df = transactions_df.merge(
    finalized_df, on="transaction_id", how="left", suffixes=("", "_val")
)

output_columns = [
    "transaction_id", "spend_description", "vendor", "source_type",
    "l1", "l2", "l3", "l4",
    "classification_reason", "confidence_level",
    "validation_status", "human_review_required", "final_status",
    "retrieved_taxonomy"
]
final_df = final_df[[c for c in output_columns if c in final_df.columns]]

out_excel = "../output/final_classification.xlsx"
os.makedirs("../output", exist_ok=True)
final_df.to_excel(out_excel, index=False)
print(f"Saved final deliverable: {out_excel} ({len(final_df)} records).")

# Evaluate accuracy against verified 189-transaction gold standard
gold_standard_file = "../data/gold_standard.csv"
if os.path.exists(gold_standard_file):
    eval_report = run_accuracy_evaluation(
        predictions_path=out_excel,
        gold_standard_path=gold_standard_file
    )

Saved final deliverable: ../output/final_classification.xlsx (197 records).
SPENDKEY GENAI CLASSIFIER - ACCURACY EVALUATION REPORT
Total Transactions Evaluated: 189 (against Gold Standard)
-----------------------------------------------------------------
  Level 1 (Segment) Accuracy:     84.66%  (160/189)
  Level 2 (Family) Accuracy:      77.25%  (146/189)
  Level 3 (Category) Accuracy:    75.13%  (142/189)
  Level 4 (Commodity) Accuracy:   69.84%  (132/189)
  Full 4-Level Path Accuracy:     69.84%  (132/189)
-----------------------------------------------------------------
GOVERNANCE & TRUST METRICS:
  Auto-Accepted Precision:        79.74%  (153 items auto-passed)
  High-Confidence Precision:      79.87%  (154 items)
  Human Review Routing:           26 items properly flagged for human review
